In [ ]:
import time
import json
import os
import requests
import keyboard
import pyperclip
import pyautogui
import ctypes
from PIL import ImageGrab
import ddddocr
import io
import uiautomation as auto

# ==================== [ 全局配置区 ] ====================
OLLAMA_API_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "gemma4"  # 匹配你本地完美运行的 gemma4 模型
NOTEBOOK_FILE = "MyNotebook.md"  # 本地自动化生词本文件路径


# ==================== [ 第一层：感知能力 (Perception) ] ====================
def get_word_under_cursor():
    """核心感知：嗅探鼠标光标悬停位置的英文文本（已修复多线程 COM 初始化 Bug）"""
    try:
        # 【核心修复】显式调用 Windows API，初始化当前后台线程的 COM 库
        # 0 代表 COINIT_APARTMENTTHREADED（单线程单元模式）
        ctypes.windll.ole32.CoInitialize(None)
        
        # 1. 尝试利用微软官方 UIA 接口直接探测底层 UI 控件
        element = auto.ControlFromCursor()
        if not element:
            return _fallback_double_click_capture()

        # 2. 尝试提取文本控制流模式
        text_pattern = element.GetTextPattern()
        if not text_pattern:
            return _fallback_double_click_capture()

        # 3. 如果控件支持原生 Text 探测，直接提取当前光标所在的文本
        selections = text_pattern.GetSelection()
        if selections and len(selections) > 0:
            return selections[0].GetText(100).strip()
            
        return _fallback_double_click_capture()

    except Exception:
        # 遇到任何异常或无响应，降级使用模拟极速双击复制
        return _fallback_double_click_capture()
        
    finally:
        # 【规范清理】无论成功与否，最后都要逆向注销 COM，防止内存泄漏
        try:
            ctypes.windll.ole32.CoUninitialize()
        except:
            pass


# 初始化本地高精度 OCR 引擎（只在启动时加载一次，保证查词响应速度）
ocr = ddddocr.DdddOcr(show_ad=False)

def _fallback_double_click_capture():
    """硬核降级感知：当 UIA 抓不到文字时（如 PDF 或图片），截取鼠标周围微小区域进行 OCR 识字"""
    global ocr
    try:
        # 1. 瞬间抓取当前鼠标在屏幕上的绝对位置
        x, y = pyautogui.position()
        
        # 2. 划定一个微小的“狙击镜”区域（以鼠标为中心，左右延伸 80 像素，上下延伸 20 像素）
        # 这个尺寸刚好可以完整包进一个长单词或短语，同时过滤掉周围无关的文本
        bbox = (x - 80, y - 20, x + 80, y + 20)
        
        # 3. 截取该区域屏幕画面
        img = ImageGrab.grab(bbox)
        
        # 4. 将截图转为内存中的字节流，避免频繁读写硬盘拖慢速度
        img_byte_arr = io.BytesIO()
        img.save(img_byte_arr, format='PNG')
        img_bytes = img_byte_arr.getvalue()
        
        # 5. 喂给本地 OCR 引擎进行文字扣取
        res_text = ocr.classification(img_bytes).strip()
        
        # 6. 如果识别出内容，直接返回扔给大模型
        if res_text:
            return res_text
    except Exception as e:
        print(f"\n⚠️ OCR 模块取词异常: {e}")
    return None


# ==================== [ 第二层：思考能力 (Thinking) ] ====================
def call_translation_agent(text):
    """调用本地大模型，注入垂直领域 System Prompt 进行高阶语法与语境拆解"""
    
    # 注入专家提示词，确保 gemma4 能够输出非常系统化的翻译与剖析
    system_instruction = (
        "你是一个精通中英互译的顶级语言学专家，同时擅长系统化的语法结构拆解。\n"
        "请对用户输入的英文进行深度剖析，严格遵循以下规范输出：\n"
        "1. 如果输入是【单个单词或短语】：给出精准的词性与中文释义。随后，从《新概念英语》或雅思考试的视角，提供一个原汁原味的英文例句并附带中文翻译。\n"
        "2. 如果输入是【完整句子】：直接给出优雅、符合中文习惯的翻译。接着，精细拆解该句子的【核心语法结构】（如主谓宾、定语从句、状语后置等），并指出其中包含的高级词汇或固定搭配。"
    )
    
    # 构造标准的 Chat 消息体格式
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": text}
        ],
        "stream": False  # 关闭流式输出，方便我们拿到完整文本后一次性写入 Markdown
    }
    
    try:
        response = requests.post(OLLAMA_API_URL, json=payload, timeout=60)
        response.raise_for_status()
        
        # 解析 Ollama 返回的 JSON 报文
        response_data = response.json()
        return response_data['message']['content']
    except Exception as e:
        return f"Agent 思考过程中发生错误: {e}"


# ==================== [ 第三层：行动能力 (Action) ] ====================
def write_to_notebook(word, analysis):
    """持久化工具：将智能体的思考和翻译成果自动化追加到本地 Markdown 笔记本中"""
    current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
    
    log_content = f"\n## 🔍 查询时间: {current_time}\n"
    log_content += f"- **查询原文**: `{word}`\n"
    log_content += f"- **智能体分析**:\n{analysis}\n"
    log_content += f"{'-'*40}\n"
    
    # 使用 utf-8 编码追加写入文件，彻底防止 Windows 平台下的中文乱码问题
    with open(NOTEBOOK_FILE, "a", encoding="utf-8") as f:
        f.write(log_content)
    
    print(f"📖 [行动] 成功将单词 '{word}' 的分析写入本地生词本: {os.path.abspath(NOTEBOOK_FILE)}")


# ==================== [ 核心驱动流 (Agent Loop) ] ====================
def main_agent_loop():
    print("====== 🦾 专属悬停翻译 Agent 已成功激活 ======")
    print(f"当前绑定大模型大脑: {MODEL_NAME}")
    print("使用方法: 将鼠标指针停留在任意英文单词上，按下 [ Ctrl + Alt + F ] 触发嗅探")
    print("安全退出: 在控制台按下 Ctrl + C 即可")
    
    def on_trigger():
        # 1. 触发感知
        word = get_word_under_cursor()
        if not word or len(word) > 40: # 如果吸出整段长文本，说明没有指对单个词
            print("⚠️ 未能在鼠标光标下精准嗅探到有效英文单词，请重新对准。")
            return
            
        print(f"\n🚀 [感知] 成功嗅探到目标单词: '{word}'，正在唤醒本地大模型...")
        
        # 2. 触发思考
        analysis_result = call_translation_agent(word)
        
        # 3. 终端实时预览
        print("\n💡 [大脑输出] 翻译与语法分析结果如下:")
        print(analysis_result)
        
        # 4. 触发行动
        write_to_notebook(word, analysis_result)
        print("="*60)

    # 绑定全局快捷键触发器
    keyboard.add_hotkey('f2', on_trigger)
    
    # 挂起主线程，保持后台监听
    keyboard.wait()


if __name__ == "__main__":
    main_agent_loop()

====== 🦾 专属悬停翻译 Agent 已成功激活 ======
当前绑定大模型大脑: gemma4
使用方法: 将鼠标指针停留在任意英文单词上，按下 [ Ctrl + Alt + F ] 触发嗅探
安全退出: 在控制台按下 Ctrl + C 即可

🚀 [感知] 成功嗅探到目标单词: 'yourhandsheon'，正在唤醒本地大模型...


In [ ]:
!pip install requests keyboard pyperclip pyautogui uiautomation

In [1]:
!pip install ddddocr pillow

^C
   ---------------------------------------- 0.0/76.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/76.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/76.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/76.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/76.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/76.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/76.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/76.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/76.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/76.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/76.0 MB 331.2 kB/s eta 0:03:48
   ---------------------------------------- 0.5/76.0 MB 331.2 kB/s eta 0:03:48
   ---------------------------------------- 0.5/76.0 MB 331.2 kB/s eta 0:03:48
   ---------------------------------------- 0.8/76.0 MB 376.


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install ddddocr -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
